# 크라베오 AI 재학습 — v7 (8B 베이스 모델 + 정체성 데이터 80개)
# Cravveo AI Retrain — v7 (8B base model + 80 identity samples)

**배경 | Background:**
Day 043에서 v2/v4/v5/v6 전부 정체성 평가세트(30문항)에 실패함을 확인.
Day 044에 두 가지 준비: (1) 정체성 데이터 20개 → 80개(같은 사실을 3~4개 표현으로 확장, 복제 아님),
(2) Llama 3.1 8B가 무료 T4에서 로드 가능하고 3B보다 훨씬 안정적인 텍스트를 생성함을 확인.
Day 043 confirmed v2/v4/v5/v6 all fail the 30-question identity eval set.
Day 044 prepared: (1) 20→80 identity samples (natural phrasing variation, not duplication),
(2) confirmed Llama 3.1 8B loads on free T4 and generates far more stable text than 3B.

**이번 학습 | This run:**
- 베이스 모델: Llama 3.1 8B Instruct (실패 시 Qwen 2.5 7B로 자동 대체)
- 데이터: 정체성 80개 + Obsidian 203개("쇼핑몰" 오류 3건 제거된 정리본) = 총 283개
- Base model: Llama 3.1 8B Instruct (auto-fallback to Qwen 2.5 7B if it fails)
- Data: 80 identity + 203 Obsidian (cleaned, 3 "online shop" errors removed) = 283 total

**순서 | Steps:**
1. 설치
2. 모델 로드 (8B, 실패시 7B)
3. LoRA 어댑터
4. 데이터셋 로드 + 합치기 (복제 없음)
5. 학습
6. 정체성 평가세트 일부로 테스트
7. HuggingFace 업로드
8. GGUF 변환 (q4_K_M)


In [ ]:
# 셀 1: 필수 라이브러리 설치
# Cell 1: Install required libraries
!pip install unsloth trl datasets


In [ ]:
# 셀 2: CUDA 오류 방지 + 모델 로드 (8B, 실패시 7B로 대체)
# Cell 2: CUDA fix + model load (8B, fallback to 7B on failure)
import ctypes, glob

paths = (
    glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/libnvJitLink*.so*") +
    glob.glob("/usr/lib/**/libnvJitLink*.so*", recursive=True) +
    glob.glob("/usr/local/cuda*/**/libnvJitLink*.so*", recursive=True)
)
if paths:
    ctypes.CDLL(paths[0], mode=ctypes.RTLD_GLOBAL)
    print(f"CUDA fix 적용 | CUDA fix applied: {paths[0]}")

from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct"
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        load_in_4bit=True,
    )
except Exception as e:
    print(f"8B 로드 실패, 7B로 재시도 | 8B failed, retrying with 7B: {e}")
    MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        load_in_4bit=True,
    )

print(f"모델 로드 완료 | Model loaded: {MODEL_NAME}")


In [ ]:
# 셀 3: LoRA 어댑터 설정
# Cell 3: LoRA adapter setup
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("LoRA 어댑터 적용 완료 | LoRA adapter applied")


In [ ]:
# 셀 4: 데이터셋 로드 + 합치기 (복제 없음)
# Cell 4: Load datasets + combine (no duplication)
from datasets import load_dataset, Dataset, concatenate_datasets
import json

# --- 정체성 데이터셋 (HuggingFace, Day 044에 80개로 확장됨) ---
existing_raw = load_dataset("cravveo/cravveo-briefing-dataset", split="train")

def convert_existing(example):
    return {"text": f"질문: {example['instruction']}\n답변: {example['output']}"}

existing = existing_raw.map(convert_existing, remove_columns=existing_raw.column_names)
print(f"정체성 데이터셋: {len(existing)}개 (기대값: 80개)")

# --- Obsidian 데이터셋 (파일 업로드, "쇼핑몰" 오류 3건 제거된 정리본) ---
# --- Obsidian dataset (upload, cleaned version with 3 "online shop" errors removed) ---
from google.colab import files
print("obsidian_dataset_cleaned.jsonl 파일을 업로드하세요 | Upload obsidian_dataset_cleaned.jsonl")
uploaded = files.upload()

new_data = []
filename = list(uploaded.keys())[0]
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        new_data.append(json.loads(line.strip()))

new_dataset = Dataset.from_list(new_data)
print(f"Obsidian 데이터셋: {len(new_dataset)}개")

# --- 합치기 (복제 없음, 셔플만) ---
combined = concatenate_datasets([existing, new_dataset])
combined = combined.shuffle(seed=42)
print(f"\n합계 | Total: {len(combined)}개 (정체성 {len(existing)} : Obsidian {len(new_dataset)} = {len(existing)/len(combined)*100:.0f}%:{len(new_dataset)/len(combined)*100:.0f}%)")
print("샘플 확인 | Sample check:")
print(combined[0]['text'])


In [ ]:
# 셀 5: 학습
# Cell 5: Training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined,
    dataset_text_field="text",
    max_seq_length=512,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="cravveo_v7_output",
        save_strategy="no",
    ),
)

print("학습 시작 | Training start...")
trainer.train()
print("학습 완료 | Training complete!")


In [ ]:
# 셀 6: 정체성 평가세트 일부로 테스트 (정체성_평가세트.md 참고)
# Cell 6: Quick test with a sample from the identity eval set
FastLanguageModel.for_inference(model)

test_questions = [
    "크라베오 컴퍼니가 뭐야?",       # A1
    "ablescan.app은 뭐야?",         # A14
    "지금 수익이 나고 있어?",        # A16
    "크라베오가 뭐 하는 곳이야?",     # B1 (표현 변형)
    "웹 접근성 검사해주는 도구 이름이 뭐야?",  # B6 (표현 변형)
]
for q in test_questions:
    inputs = tokenizer([f"질문: {q}\n답변: "], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.2)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("---")

print("\n전체 30문항 채점은 GGUF 변환 후 Ollama에서 정체성_평가세트.md 기준으로 진행할 것")


In [ ]:
# 셀 6.5: 정체성 평가세트 30문항 전체 테스트 (GGUF 변환 전에 먼저 확인)
# Cell 6.5: Full 30-question identity eval BEFORE GGUF export
questions = [
    "크라베오 컴퍼니가 뭐야?", "크라베오 컴퍼니의 운영자는 누구인가요?", "100일 AI 도전이 뭐야?",
    "Build in Public이 뭐야?", "왜 1인 개발을 선택했어?", "파인튜닝을 왜 배우고 있어?",
    "크라베오의 목표가 뭐야?", "유튜브를 왜 시작했어?", "지금 파인튜닝 중인 모델은 뭐야?",
    "파인튜닝에 어떤 도구를 쓰고 있어?", "LoRA가 뭐야?", "GPU 없이도 파인튜닝이 가능해?",
    "Chrome 확장 프로그램은 어떤 걸 만들었어?", "ablescan.app은 뭐야?", "cravveo.com은 뭐야?",
    "지금 수익이 나고 있어?", "100일 도전 중 가장 어려웠던 게 뭐야?", "유튜브 채널에서는 어떤 내용을 다뤄?",
    "크라베오 AI 어시스턴트는 어떤 역할을 할 예정이야?", "하루를 어떻게 보내?",
    "크라베오가 뭐 하는 곳이야?", "누가 크라베오를 운영해?", "100일 챌린지가 무슨 프로젝트야?",
    "실패한 것도 숨기지 않고 다 보여주는 방식을 뭐라고 해?", "돈은 좀 벌고 있어?",
    "웹 접근성 검사해주는 도구 이름이 뭐야?", "파인튜닝할 때 어댑터만 학습하는 기술 이름이 뭐야?",
    "지금까지 게시한 크롬 확장프로그램이 몇 개야?", "노트북에 그래픽카드 없는데 어떻게 학습해?",
    "왜 혼자 일하기로 했어?",
]
labels = [f"A{i+1}" for i in range(20)] + [f"B{i+1}" for i in range(10)]

for label, q in zip(labels, questions):
    inputs = tokenizer([f"질문: {q}\n답변: "], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=120, temperature=0.2)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"[{label}] {q}")
    print(answer)
    print("---")


In [ ]:
# 셀 7: HuggingFace 업로드
# Cell 7: Push to HuggingFace
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
login(token=token)

model.save_pretrained("cravveo-llama-lora-v7")
model.push_to_hub("cravveo/cravveo-llama-lora", token=token)  # 같은 레포에 덮어쓰기
tokenizer.push_to_hub("cravveo/cravveo-llama-lora", token=token)

print("HuggingFace 업로드 완료 | HuggingFace upload complete")
print("https://huggingface.co/cravveo/cravveo-llama-lora")


In [ ]:
# 셀 8: GGUF 변환 (q4_K_M)
# Cell 8: GGUF conversion (q4_K_M)
model.save_pretrained_gguf(
    "cravveo-v7-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

from google.colab import files
import os

folder = "cravveo-v7-gguf_gguf" if os.path.isdir("cravveo-v7-gguf_gguf") else "cravveo-v7-gguf"
gguf_file = [f for f in os.listdir(folder) if f.endswith(".gguf")][0]
size_gb = os.path.getsize(f"{folder}/{gguf_file}") / 1e9

print(f"파일명: {gguf_file}")
print(f"파일 크기: {size_gb:.2f} GB")

files.download(f"{folder}/{gguf_file}")
print("다운로드 시작 | Download started")
